In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
# Permite importar tu paquete de src/ sin instalarlo
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))


# 06 · Optimización con seguimiento (tracking)

> **Esqueleto.** Guía completa: `curso/kit-estudiante/guia-modelado-y-optimizacion.md`, secciones 3 y 4.

Buscar mejores hiperparámetros es probar muchas configuraciones. Sin registro, a la mitad ya no sabes
cuál era la buena: por eso cada ensayo queda en **MLflow**. Tres cuidados:

1. Se optimizan **solo los finalistas** del torneo.
2. El **espacio de búsqueda** lleva tu conocimiento del modelo, y el **presupuesto** de ensayos se fija antes.
3. La mejor puntuación de una búsqueda es **optimista**: se **comprueba** en entidades que la búsqueda no vio.

## 0. Cargar los datos de entrenamiento

In [ ]:
import pandas as pd

# === Configuración: completa estas dos líneas (las mismas de tu notebook 03) ===
ID_COL = None     # TODO: columna que identifica a la entidad que se repite (None si no hay entidades repetidas)
TARGET = None     # TODO: tu columna objetivo

train_path = Path.cwd().resolve().parent / "data" / "processed" / "train.joblib"
if not train_path.exists():
    train = None
    print("Todavía no hay datos de entrenamiento: completa y ejecuta antes el notebook 03_preprocesamiento.")
else:
    import joblib
    train = joblib.load(train_path)
    print(f"entrenamiento: {train.shape[0]:,} filas x {train.shape[1]} columnas (la prueba NO se abre hoy)")

listo = train is not None and TARGET is not None
if train is not None and TARGET is None:
    print("Define TARGET (y, si hay entidades repetidas, ID_COL) arriba para continuar.")

## 1. Pipeline y métrica (los del torneo)

Importa tu función de `src/proyecto/models/train.py` y define la **misma métrica principal** del torneo: la
optimización se mide con la métrica con la que elegiste a tus finalistas.

In [ ]:
try:
    from proyecto.models.train import build_pipeline   # TODO: crea build_pipeline(modelo, ...) en src/proyecto/models/train.py
except ModuleNotFoundError:
    build_pipeline = None
    print("TODO: crea src/proyecto/models/train.py con build_pipeline(modelo): preprocesamiento + modelo dentro de un Pipeline.")

scoring = None   # TODO: dict de métricas, p. ej. {"costo": make_scorer(tu_costo, greater_is_better=False), "f1": "f1_macro"}
METRICA = None   # TODO: nombre de tu métrica principal dentro de scoring (si es negativa, multiplícala por -1 al leerla)

## 2. Conexión con el seguimiento

Usa el módulo de la plantilla en modo **local**: en esta clase no hace falta Databricks (entra en la clase 4).

In [ ]:
try:
    from proyecto.tracking.mlflow_utils import configure_mlflow, log_run, set_experiment
except ModuleNotFoundError:
    configure_mlflow = None
    print("TODO: el módulo src/proyecto/tracking/mlflow_utils.py viene en la plantilla; si no está, vuelve a clonar la rama base.")

if configure_mlflow is None:
    donde = experimento = None
else:
    with warnings.catch_warnings(record=True) as avisos:   # mostramos por qué se usó (o no) Databricks
        warnings.simplefilter("always")
        donde = configure_mlflow(force_local=True)   # clase 3: seguimiento local
    for aviso in avisos:
        print("Aviso:", aviso.message)
    experimento = set_experiment("mi-proyecto-optimizacion")   # TODO: cambia "mi-proyecto" por el nombre de tu proyecto
    print("MLflow:", donde, "| experimento:", experimento)

> **En esta clase el seguimiento es local:** todo se guarda en `mlflow_local.db` (raíz del repo, ignorado por Git).
> Para verlo en el navegador: `mlflow ui --backend-store-uri sqlite:///mlflow_local.db`. **Databricks entra en la clase 4.**

## 3. Reservar entidades que la búsqueda no verá

Separa, **dentro del entrenamiento**, un 25 % de las **entidades** (no de las filas). La búsqueda trabaja solo
con el resto; `test.joblib` sigue cerrado.

In [ ]:
if not listo:
    busqueda = reservado = None
    print("Sin datos o sin TARGET: esta celda separará las entidades reservadas cuando estén.")
elif ID_COL is None:
    from sklearn.model_selection import train_test_split
    busqueda, reservado = train_test_split(train, test_size=0.25, random_state=7)
    print("Sin entidades repetidas: se reserva por fila (explica por qué es válido en tu caso).")
else:
    from sklearn.model_selection import GroupShuffleSplit

    i_busqueda, i_reservado = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7).split(train, groups=train[ID_COL]))
    busqueda, reservado = train.iloc[i_busqueda], train.iloc[i_reservado]
    assert set(busqueda[ID_COL]).isdisjoint(set(reservado[ID_COL])), "hay entidades en los dos lados"
    print(f"búsqueda: {busqueda[ID_COL].nunique():,} entidades · reservadas: {reservado[ID_COL].nunique():,} entidades")

## 4. Espacio de búsqueda y presupuesto

Acota los rangos con lo que sabes del algoritmo (ver la tabla de la sección 3.2 de la guía) y fija el
**número de ensayos antes de empezar**.

In [ ]:
N_ENSAYOS = 12          # presupuesto: cada ensayo cuesta tiempo y memoria
FINALISTA = None        # TODO: el nombre de tu finalista del torneo

def espacio(trial):
    """TODO: devuelve un dict con los hiperparámetros del ensayo, p. ej.
    {"learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True), ...}"""
    return None

print("Presupuesto:", N_ENSAYOS, "ensayos · finalista:", FINALISTA or "TODO")

## 5. La búsqueda (un ensayo = una ejecución registrada)

Cada ensayo se registra con `log_run`: parámetros, métrica principal y etiquetas.

In [ ]:
estudio = None
pendientes = [nombre for nombre, valor in {"datos de búsqueda": busqueda, "build_pipeline": build_pipeline,
                                              "scoring": scoring, "FINALISTA": FINALISTA, "tracking": configure_mlflow}.items()
              if valor is None]
if pendientes:
    print("Pendiente: completa", ", ".join(pendientes), "para lanzar la búsqueda.")
else:
    print("TODO: aquí va la búsqueda (por ejemplo con Optuna): define `objetivo(trial)`, ejecútalo",
          N_ENSAYOS, "veces y registra cada ensayo con log_run(...). Pídele a Claude el código con tu espacio.")

## 6. La comprobación honesta

Entrena **dos** modelos con los datos de la búsqueda (parámetros de **partida** y **optimizados**) y compáralos
**en las entidades reservadas**, con un intervalo del 95 % que remuestrea **entidades completas**, no filas.

In [ ]:
import numpy as np


def intervalo_por_entidad(diferencia_por_fila, entidades, n=1000, seed=42):
    """diferencia_por_fila = costo(optimizado) - costo(partida), fila a fila. Negativo: el optimizado mejora."""
    por_entidad = pd.DataFrame({"g": np.asarray(entidades), "d": np.asarray(diferencia_por_fila)}).groupby("g")["d"].agg(["sum", "count"])
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(por_entidad), size=(n, len(por_entidad)))
    medias = por_entidad["sum"].to_numpy()[idx].sum(axis=1) / por_entidad["count"].to_numpy()[idx].sum(axis=1)
    return np.percentile(medias, [2.5, 97.5])


# TODO: calcula `diferencia_por_fila` en `reservado` con tus dos modelos y llama a intervalo_por_entidad(...)
bajo = alto = None
if bajo is None:
    print("Pendiente: ajusta los dos modelos, calcula la diferencia por fila en las entidades reservadas y obtén el intervalo.")
else:
    print(f"Intervalo del 95 %: [{bajo:+.4f}, {alto:+.4f}]")

**Regla de decisión escrita ANTES de mirar:** TODO. Ejemplo: *uso los parámetros optimizados solo si todo el
intervalo está por debajo de cero; si cruza el cero, la mejora no se distingue del ruido y uso los de partida.*

**Resultado:** TODO (intervalo, y qué parámetros usas a partir de ahora, y por qué).

## 7. Guardar la decisión

In [ ]:
import json

mejores_parametros = None   # TODO: dict con los parámetros que decidiste usar (de partida u optimizados)
if mejores_parametros is None:
    print("Sin decisión todavía: esta celda guardará reports/06_best_params.json cuando la tomes.")
else:
    salida = Path.cwd().resolve().parent / "reports" / "06_best_params.json"
    registro = {"finalista": FINALISTA, "parametros": mejores_parametros, "tracking": donde,
                "intervalo_95": [float(bajo), float(alto)] if bajo is not None else None}
    salida.write_text(json.dumps(registro, indent=2, default=float), encoding="utf-8")
    print("guardado:", salida.name)

## 8. Comprobar el seguimiento

Abre tu experimento con `mlflow ui --backend-store-uri
sqlite:///mlflow_local.db`: ¿están **todos** los ensayos? ¿cada uno con sus parámetros y métricas? ¿puedes
ordenarlos por la métrica principal?

## Entregable de esta etapa

- [ ] **Un ensayo = una ejecución registrada**, con parámetros, métrica y etiquetas.
- [ ] El **presupuesto** de ensayos está escrito y se respetó.
- [ ] Hay **entidades reservadas** que la búsqueda nunca vio.
- [ ] La **comprobación** (intervalo del 95 % remuestreando entidades) está hecha y la regla de decisión
  estaba escrita antes.
- [ ] `reports/06_optimizacion.md` y `reports/06_best_params.json` completados.